# Uniform Cost Search: Nairobi to Kisumu

**Self-driving trailer route-finding exercise**

This notebook models the Part 1 directed highway graph and applies Uniform Cost Search (UCS) to find the least-cost route from Nairobi to Kisumu. Edge costs are the distances supplied in the exercise, in kilometres. UCS expands the frontier node with the lowest cumulative path cost, it stops when the goal is removed from the priority queue.



## 1. Define the search problem

- **Initial state:** Nairobi
- **Goal state:** Kisumu
- **States:** Towns in the directed graph
- **Actions:** Travel along an outgoing highway edge
- **Step cost:** Distance of that edge in kilometres
- **Path cost:** Sum of step costs from Nairobi
- **Goal test:** Current town is Kisumu

The directed connections and distances are transcribed from Part 1. An edge is traversable only in the direction listed.


In [1]:
from heapq import heappush, heappop
from itertools import count

# Adjacency list: each (neighbour, distance_km) is a directed edge.
graph = {
    "Nairobi": [("Nakuru", 161), ("Naivasha", 93), ("Nyeri", 151)],
    "Naivasha": [("Nakuru", 70)],
    "Nyeri": [("Nyahururu", 104)],
    "Nyahururu": [("Nakuru", 64)],
    "Nakuru": [("Kericho", 107), ("Eldoret", 171)],
    "Kericho": [("Kisumu", 84)],
    "Eldoret": [("Kisumu", 139)],
    "Kisumu": [],
}

start = "Nairobi"
goal = "Kisumu"


## 2. Run Uniform Cost Search

The priority queue is ordered by accumulated distance. A counter breaks ties deterministically without affecting path cost. The `best_cost` table prevents a more expensive route to the same town from replacing a cheaper one. The printed trace shows each town expanded and the frontier costs after its outgoing edges are considered.


In [2]:
def uniform_cost_search(graph, start, goal):
    tie_breaker = count()
    frontier = [(0, next(tie_breaker), start, [start])]
    best_cost = {start: 0}
    expansion_trace = []

    while frontier:
        cost, _, town, path = heappop(frontier)

        # Ignore stale queue entries if a cheaper route was found later.
        if cost != best_cost.get(town):
            continue

        expansion_trace.append({"expanded": town, "cost": cost})

        # UCS is optimal for nonnegative edge costs when the goal is popped.
        if town == goal:
            return path, cost, expansion_trace

        for neighbour, step_cost in graph.get(town, []):
            new_cost = cost + step_cost
            if new_cost < best_cost.get(neighbour, float("inf")):
                best_cost[neighbour] = new_cost
                heappush(
                    frontier,
                    (new_cost, next(tie_breaker), neighbour, path + [neighbour]),
                )

    return None, float("inf"), expansion_trace

path, total_cost, trace = uniform_cost_search(graph, start, goal)

print("Expansion order (town, cumulative cost):")
for step in trace:
    print(f"  {step['expanded']:<10} {step['cost']:>3} km")


Expansion order (town, cumulative cost):
  Nairobi      0 km
  Naivasha    93 km
  Nyeri      151 km
  Nakuru     161 km
  Nyahururu  255 km
  Kericho    268 km
  Eldoret    332 km
  Kisumu     352 km


## 3. Optimal route and attached path cost

The route cost below is recomputed from the graph edges so that the displayed total is tied directly to the selected path.


In [3]:
if path is None:
    print(f"No route was found from {start} to {goal}.")
else:
    edge_costs = [
        next(distance for neighbour, distance in graph[a] if neighbour == b)
        for a, b in zip(path, path[1:])
    ]
    print("Optimal path:", " → ".join(path))
    print("Edge costs (km):", " + ".join(map(str, edge_costs)))
    print(f"Total path cost: {sum(edge_costs)} km")
    assert sum(edge_costs) == total_cost


Optimal path: Nairobi → Nakuru → Kericho → Kisumu
Edge costs (km): 161 + 107 + 84
Total path cost: 352 km


## Result

For the supplied directed graph, UCS returns **Nairobi → Nakuru → Kericho → Kisumu**, with a total path cost of **352 km** (161 + 107 + 84). This agrees with the optimal route identified in Part 1.


